# Step 1 (ASMR): Trial-level NLL with a Centaur proxy

This notebook computes trial-level negative log-likelihoods from `Data/Preprocessed Data/narrative_data_food.csv`.

It uses a small LLM (`Qwen/Qwen2.5-1.5B-Instruct`) as a local Centaur proxy for testing.

## What this notebook does

- Loads participant narratives already prepared for the Centaur-style setup.
- Keeps training trials in the prompt as context (`[TRAIN]`), so the model can use learned information.
- Runs a language model forward pass over each participant narrative.
- Computes token-level negative log-likelihood (NLL).
- Aggregates token NLL over each response span `<< ... >>`, but keeps only `[TEST]` trials for final output.
- Saves outputs for downstream ASMR steps.

This is Step 1 only (Centaur-side trial likelihood extraction).

In [3]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
# Easy-to-edit configuration
INPUT_CSV        = Path("../Data/Preprocessed Data/narrative_data_food.csv")
OUTPUT_DIR       = Path("../Data/Model Outputs/centaur_step1")
MODEL_NAME       = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_PARTICIPANTS = 3  # quick local test
MAX_SEQ_LEN      = 4096
FULL_RUN         = False

ESTIMATE_PATTERN = re.compile(r"<<(.*?)>>", re.DOTALL) # Find all substrings wrapped in << >>, even if they span multiple lines, and extract what’s inside.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Configuration notes

- Start with `FULL_RUN = False` and small `MAX_PARTICIPANTS` to verify everything works.
- If GPU is available, the model loads in 4-bit for lower memory use.
- After local validation, set `FULL_RUN = True` or move this notebook/script to your cluster.

In [ ]:
df       = pd.read_csv(INPUT_CSV)[["ID", "narrative"]].copy()
df["ID"] = df["ID"].astype(str)
df       = df.sort_values("ID").reset_index(drop=True)

if not FULL_RUN:
    df = df.head(MAX_PARTICIPANTS)

print("Participants to process:", len(df))
df.head(2)

Participants to process: 3


,ID,narrative
0,02m3zllu1k3s,Your task is to estimate the carbohydrate cont...
1,0ba3662lv6vw,Your task is to estimate the carbohydrate cont...


In [8]:
device    = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

kwargs = {"device_map": "auto"} if device == "cuda" else {}
if device == "cuda":
    kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kwargs)
if device == "cpu":
    model.to("cpu")
model.eval()

print("Device:", device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

c:\Users\dizydorc\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dizydorc\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device: cpu


In [ ]:
def estimate_spans(text):
    spans = []
    cursor = 0
    trial_idx = 1
    for line in text.splitlines(keepends=True):
        phase = "unknown"
        if "[TRAIN]" in line:
            phase = "training"
        elif "[TEST]" in line:
            phase = "testing"

        for m in ESTIMATE_PATTERN.finditer(line):
            spans.append({
                "trial_index": trial_idx,
                "phase": phase,
                "estimate_text": m.group(1).strip(),
                "start": cursor + m.start(1),
                "end": cursor + m.end(1),
            })
            trial_idx += 1
        cursor += len(line)
    return spans

def token_positions_for_span(offsets, start_char, end_char):
    keep = []
    for i, (a, b) in enumerate(offsets):
        if b <= a:
            continue
        if b <= start_char:
            continue
        if a >= end_char:
            break
        keep.append(i)
    return keep

def participant_trial_nlls(participant_id, text):
    spans = estimate_spans(text)
    enc = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )
    input_ids = enc["input_ids"].to(device)
    attn = enc["attention_mask"].to(device)
    offsets = enc["offset_mapping"][0].tolist()

    with torch.no_grad():
        out = model(input_ids=input_ids, attention_mask=attn)

    logits = out.logits[0, :-1, :].cpu()
    targets = input_ids[0, 1:].cpu()
    token_nll = F.cross_entropy(logits, targets, reduction="none").numpy()

    rows = []
    for s in spans:
        token_ids = token_positions_for_span(offsets, s["start"], s["end"])
        loss_ids = [i - 1 for i in token_ids if i > 0 and (i - 1) < len(token_nll)]
        nll = float(np.sum(token_nll[loss_ids])) if loss_ids else np.nan
        rows.append({
            "participant_id": participant_id,
            "trial_index": s["trial_index"],
            "phase": s["phase"],
            "estimate_text": s["estimate_text"],
            "trial_nll": nll,
            "num_tokens": len(loss_ids),
        })
    return rows

## Quick diagnostics

This cell gives an immediate sanity check of output quality and distribution.

In [ ]:
import matplotlib.pyplot as plt

print('Rows:', len(trial_df))
print('Participants:', trial_df['participant_id'].nunique())
print('Missing trial_nll:', trial_df['trial_nll'].isna().sum())

ax = trial_df['trial_nll'].dropna().hist(bins=40)
ax.set_title('Distribution of trial-level NLL')
ax.set_xlabel('trial_nll')
ax.set_ylabel('count')
plt.show()

summary_df.head()

In [ ]:
rows = []
for i, r in df.iterrows():
    print(f"[{i+1}/{len(df)}] {r['ID']}")
    rows.extend(participant_trial_nlls(r["ID"], r["narrative"]))
    if device == "cuda":
        torch.cuda.empty_cache()

trial_df = pd.DataFrame(rows)
trial_df = trial_df[trial_df["phase"] == "testing"].copy()  # score only testing phase

summary_df = (
    trial_df.groupby("participant_id", as_index=False)
    .agg(total_trial_nll=("trial_nll", "sum"), mean_trial_nll=("trial_nll", "mean"), n_trials=("trial_index", "count"))
)

trial_df["model_name"] = MODEL_NAME
trial_df.to_csv(OUTPUT_DIR / "trial_nlls.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "participant_summary.csv", index=False)

print("Saved:", OUTPUT_DIR / "trial_nlls.csv")
print("Saved:", OUTPUT_DIR / "participant_summary.csv")
trial_df.head()